In [4]:
import pandas as pd

In [5]:
data = pd.read_csv('./../tmpsnx71rnm.csv')
data.columns

Index(['Unnamed: 0', 'Rank (Borda)', 'Model', 'Zero-shot',
       'Active Parameters (B)', 'Total Parameters (B)', 'Embedding Dimensions',
       'Max Tokens', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining',
       'Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'Reranking',
       'Retrieval', 'STS'],
      dtype='str')

In [6]:
data.drop(columns=['Unnamed: 0', 'Zero-shot','Active Parameters (B)','Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'STS', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining'], inplace=True)

In [7]:
data['Model'] = data['Model'].apply(lambda x: x.split('(')[0].strip().replace('[', '').replace(']', ''))

In [8]:
data.isnull().sum()

Rank (Borda)              0
Model                     0
Total Parameters (B)     48
Embedding Dimensions     16
Max Tokens               35
Reranking               237
Retrieval               234
dtype: int64

In [9]:
for col in data.columns:
    data[col] = data[col].fillna('0.00')
    if data[col].dtype == 'object':
        data[col]=data[col].astype(float)


In [10]:
filtered = data[
    (data['Total Parameters (B)'] != 0.00) &
    (data['Total Parameters (B)'] < 0.25)# & (data['Total Parameters (B)'] < 2.0)
]
sorted_data = filtered.sort_values('Total Parameters (B)', ascending=True)


In [11]:
sorted_data= sorted_data.sort_values('Retrieval', ascending=False)

In [12]:
sorted_data.head(10)

,Rank (Borda),Model,Total Parameters (B),Embedding Dimensions,Max Tokens,Reranking,Retrieval
19,19,jina-embeddings-v5-text-nano,0.212,768.0,8192.0,64.63,63.26
82,83,granite-embedding-97m-multilingual-r2,0.097,384.0,8192.0,59.39,60.32
57,58,F2LLM-v2-160M,0.159,640.0,40960.0,60.34,54.08
65,66,multilingual-e5-small,0.118,384.0,512.0,60.43,50.91
68,69,F2LLM-v2-80M,0.080,320.0,40960.0,58.95,50.13
59,60,bilingual-embedding-small,0.118,384.0,512.0,59.31,49.55
83,84,granite-embedding-107m-multilingual,0.107,384.0,512.0,58.48,48.08
113,114,static-similarity-mrl-multilingual-v1,0.108,1024.0,0.0,49.45,41.21
87,88,nomic-embed-text-v1-ablated,0.137,768.0,8192.0,45.04,40.74
98,99,nomic-embed-text-v1-unsupervised,0.137,768.0,8192.0,48.18,40.66


In [13]:
sorted_data.to_csv('./../filtered_sorted_models.csv', index=False)

In [14]:
import os
os.chdir('./..')

In [15]:
from src.utils import log, CustomException
log = log()

1. Based on research I chose bge-base-en-v1.5.
Features:
    - 0.1B parameters 
    - model size is 430MB approx.
    - 512 MAX tokens
    - Retriever effieciency is also good.

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
tokenizer = model.tokenizer

/home/vraj/.conda/envs/eu-mdr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14827.44it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
text = "What is the capital of France? The capital of France is Paris. You can also visit the Eiffel Tower in Paris."
token_count = tokenizer.encode(text)
print(f"Number of tokens: {len(token_count)}")

Number of tokens: 28


In [21]:
import sys
import json
try:
    log.info("Loading cleaned EU MDR 2017-745 documents for token length analysis.")
    with open('data/processed/cleaned_eu_mdr_2017-745.json', 'r') as f:
        docs = json.load(f)
    
    structure_prefix = ""
    part_pattern = "\nPART [A-Z] \n"
    simple_pattern = r'\n(\d+)\.\s*\n'
    decimal_pattern = "\n\d+\.\d+\.\s*\n"
    triple = "\n\d+\.\d+\.\d+\.\s*\n"

    for i, doc in enumerate(docs):
        page_content = doc.get('page_content')
        metadata = doc.get('metadata')
        token_length = len(tokenizer.encode(page_content))
      

            
except Exception as e:
    log.exception(f"An error occurred: {e}")
    raise CustomException(e, sys)

2026-06-03 11:17:41,282 53 3204945777 - INFO - Loading cleaned EU MDR 2017-745 documents for token length analysis.
Token indices sequence length is longer than the specified maximum sequence length for this model (11358 > 512). Running this sequence through the model will result in indexing errors


In [22]:
for doc in docs:
    doc.get('page_content', '')

In [119]:
info = docs[148].get('page_content', '')

In [120]:
import re
pattern = r'\n(\d+)\.\s*\n(.+)'
part_pattern = "\nPART [A-Z] \n(.+)"
matches =  re.finditer(part_pattern, docs[147].get('page_content'))

In [121]:
len(tokenizer.encode(docs[147].get('page_content', '')))

4807

In [122]:
patterns = {"part" : "\nPART [A-Z] \n(.+)", "simple" : r'\n(\d+)\.\s*\n(.+)', "decimal" : "\n\d+\.\d+\.\s*\n(.+)",
"triple" : "\n\d+\.\d+\.\d+\.\s*"}
def token_length(page_content):
    return len(tokenizer.encode(page_content))
def get_split_levels(page_content):
    levels = []
    if token_length(page_content) <= 512:
        return ['no_split']
    if re.search(patterns['part'], page_content):
        levels.append("part")
    if re.search(patterns['simple'], page_content):
        levels.append("simple")
    if re.search(patterns['decimal'], page_content):
        levels.append("decimal")
    if re.search(patterns['triple'], page_content):
        levels.append("triple")
    return levels

In [125]:
def get_text_piece(pattern, text):
    find = re.finditer(pattern, text, re.M)
    matches = []
    pieces = []
    for match in find:
        matches.append((match.start(), match.group()))
    if not matches:
        return [text]
    if matches and matches[0][0] > 0:
        pieces.insert(0, text[0:matches[0][0]])
    for i, mark in enumerate(matches):
        pattern_length = len(mark[1])
        if i < len(matches)-1:
            pieces.append(text[mark[0]:matches[i+1][0]])
        else:
            pieces.append(text[mark[0]:])
        
    return pieces

a = get_text_piece(patterns['simple'], info)
lengths = [token_length(piece) for piece in a]
a


['ANNEX VII \nREQUIREMENTS TO BE MET BY NOTIFIED BODIES ',
 "\n1.  \nORGANISATIONAL AND GENERAL REQUIREMENTS \n1.1.  \nLegal status and organisational structure \n1.1.1.  Each notified body shall be established under the national law of a Member State, or under the law of a third \ncountry with which the Union has concluded an agreement in this respect. Its legal personality and status shall be \nfully documented. Such documentation shall include information about ownership and the legal or natural \npersons exercising control over the notified body. \n1.1.2.  If the notified body is a legal entity that is part of a larger organisation, the activities of that organisation as well \nas its organisational structure and governance, and the relationship with the notified body shall be clearly \ndocumented. In such cases, the requirements of Section 1.2 are applicable to both the notified body and the \norganisation to which it belongs. \n1.1.3.  If a notified body wholly or partly owns leg

In [118]:
f = re.match(part_pattern, a[0]).group()
f

AttributeError: 'NoneType' object has no attribute 'group'

In [110]:
s = get_text_piece(patterns['simple'], a[-1])
s

['\nPART C \nTHE UDI SYSTEM ',
 "\n1.  \nDefinitions \nAutomatic identification and data capture ('AIDC') \nAIDC is a technology used to automatically capture data. AIDC technologies include bar codes, smart cards, \nbiometrics and RFID. \nBasic UDI-DI \nThe Basic UDI-DI is the primary identifier of a device model. It is the DI assigned at the level of the device unit \nof use. It is the main key for records in the UDI database and is referenced in relevant certificates and EU \ndeclarations of conformity. \nUnit of Use DI \nThe Unit of Use DI serves to associate the use of a device with a patient in instances in which a UDI is not \nlabelled on the individual device at the level of its unit of use, for example in the event of several units of the \nsame device being packaged together. \nConfigurable device \nA configurable device is a device that consists of several components which can be assembled by the \nmanufacturer in multiple configurations. Those individual components may be d

In [112]:
re.match(patterns['simple'],s[1]).group()

'\n1.  \nDefinitions '

In [ ]:
def create_chunks(text, metadata):
    return {'page_content': text, 'metadata': metadata}

def split_document(text, metadata, levels, prefix=""):
    if not levels and token_length(text)<=512:
        return [create_chunks(text, metadata)]
    
    current_level= levels[0]
    remaining_levels = levels[1:]

    pieces = get_text_piece(patterns[current_level], text)
    chunks = []
    
    for i, piece in enumerate(pieces):
        total_chunks = len(pieces)
        chunk_id = i+1
        metadata['total_chunk'] = total_chunks
        metadata['chunk_id'] = chunk_id
        lines = piece.strip().split('\n')
        header = "Chunk_context:" + "-".join(lines[:2])
        piece_text = "\n".join(lines[2:])
        new_prefix = prefix + "-" + header if prefix else header
        chunks.extend(split_document(piece_text, metadata, remaining_levels, new_prefix))

In [152]:
info = docs[147].get('page_content', '')
meta = docs[147].get("metadata")
level = get_split_levels(info)
x=split_document(info, meta, level)
x

prefix: 'ANNEX VI -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC ', token_length: 58, condition: True
prefix: 'PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC -PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC ', token_length: 76, condition: True
prefix: 'PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC -1.  -Information relating to the economic operator ', token_length: 108, condition: True
prefix: 'PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC -2.  -Information relating to the device ', token_length: 455, condition: True
prefix: 'PART B -CORE DATA ELEMENTS TO BE PROVIDED TO THE UDI DATABASE TOGETHER WITH THE UDI-DI IN ', token_length: 481, condition: True
prefix: 'PART C -THE UDI SYSTEM -PART C -THE UDI SYSTEM', token_length: 8, condition: False
prefix: 'PART C -THE UDI SYSTEM -1.  -Definitions -1.  -Definitions -1.  -Defin

[{'page_content': 'ANNEX VI -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nANNEX VI \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31, CORE DATA ELEMENTS TO BE PROVIDED \nTO THE UDI DATABASE TOGETHER WITH THE UDI-DI IN ACCORDANCE WITH ARTICLES 28 AND 29, \nAND THE UDI SYSTEM ',
  'metadata': {'document_name': 'eu_mdr_2017-745.pdf',
   'document_type': 'Regulations',
   'chapter': '',
   'chapter_title': '',
   'article': '',
   'article_title': '',
   'annex': 'ANNEX VI ',
   'annex_title': 'INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC ',
   'section': '',
   'section_title': '',
   'page_number': 115,
   'cross_references': '',
   'token_length': 58}},
 {'page_content': 'PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC -PART A -INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \n\nPART A \nINF

In [ ]:
pieces = get_text_piece(patterns['simple'], a[-1])
print(len(pieces))
for i, p in enumerate(pieces):
    print(f"Piece {i}: {repr(p[:50])}, tokens: {token_length(p)}")

7
Piece 0: '\nPART C \nTHE UDI SYSTEM ', tokens: 8
Piece 1: '\n1.  \nDefinitions \nAutomatic identification and da', tokens: 707
Piece 2: '\n2.  \nGeneral requirements \n2.1.  \nThe affixing of', tokens: 105
Piece 3: '\n3.  \nThe UDI \n3.1.  \nA UDI shall be assigned to t', tokens: 516
Piece 4: '\n4.  \nUDI carrier \n4.1.  \nThe UDI carrier (AIDC an', tokens: 757
Piece 5: '\n5.  \nGeneral principles of the UDI database \n5.1.', tokens: 337
Piece 6: '\n6.  \nRules for specific device types \n6.1.  \nImpl', tokens: 1221


In [72]:
idx = a[-1].find('1.')
print(repr(a[-1][idx-2:idx+20]))

' \n1.  \nDefinitions \nAu'


In [74]:
pieces = get_text_piece(patterns['simple'], a[-1])
print(len(pieces))
for i, p in enumerate(pieces):
    print(f"Piece {i}: {repr(p[:50])}, tokens: {token_length(p)}")

7
Piece 0: '\nPART C \nTHE UDI SYSTEM ', tokens: 8
Piece 1: '\n1.  \nDefinitions \nAutomatic identification and da', tokens: 707
Piece 2: '\n2.  \nGeneral requirements \n2.1.  \nThe affixing of', tokens: 105
Piece 3: '\n3.  \nThe UDI \n3.1.  \nA UDI shall be assigned to t', tokens: 516
Piece 4: '\n4.  \nUDI carrier \n4.1.  \nThe UDI carrier (AIDC an', tokens: 757
Piece 5: '\n5.  \nGeneral principles of the UDI database \n5.1.', tokens: 337
Piece 6: '\n6.  \nRules for specific device types \n6.1.  \nImpl', tokens: 1221


In [75]:
piece_1 = pieces[1]
print(token_length(piece_1))
sub_chunks = split_document(piece_1, meta, ['decimal', 'triple'])
print(len(sub_chunks))
print([c['metadata']['token_length'] for c in sub_chunks])

707
0
[]


In [76]:
print(patterns['decimal'])
print(re.findall(patterns['decimal'], piece_1, re.M))


\d+\.\d+\.\s*

[]
